# Module 3 - Intent Classifier
Uses LLM prompting (zero-shot and few-shot) via Groq API to classify 
user message intent into one of 5 categories:
greeting, goodbye, gratitude, asking_mental_health_question, out_of_scope

In [ ]:
# run this install library
#!pip install groq python-dotenv

# sanity check 

In [1]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Quick sanity check to confirm that the api key is working
test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",   # ← changed this line
    messages=[{"role": "user", "content": "say hello"}],
    max_tokens=10
)
print(test.choices[0].message.content)

Hello. How can I assist you today?


In [ ]:
# #run this cell if you want to see all the available models

# models = client.models.list()
# for m in models.data:
#     print(m.id)

# Zero shot classifier

In [6]:
def classify_intent_zero_shot(user_message: str) -> str:
    
    prompt = f"""You are an intent classification system for a mental health chatbot.

Classify the following user message into EXACTLY one of these intents:
- greeting: the user is saying hello or starting a conversation
- goodbye: the user is ending the conversation
- gratitude: the user is saying thank you or expressing appreciation  
- asking_mental_health_question: the user is asking about or describing a mental health issue, emotion, or personal struggle
- out_of_scope: the message has nothing to do with mental health or conversation (weather, sports, cooking, etc.)

Rules:
- Reply with ONLY the intent label, nothing else
- No punctuation, no explanation, just the label
- If unsure between two, pick the most likely one

User message: "{user_message}"

Intent:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # use the model u choose in the sanity check.
        messages=[{"role": "user", "content": prompt}],
        temperature=0,     # 0 = deterministic, same answer every time
        max_tokens=20      # only one short label
    )
    
    raw_output = response.choices[0].message.content.strip().lower()
    return raw_output


# Quick manual tests
tests = [
    "hello there",
    "I have been feeling really anxious lately",
    "thank you so much",
    "bye goodbye",
    "what is the best restaurant in Cairo"
]

print("=== ZERO-SHOT RESULTS ===")
for t in tests:
    result = classify_intent_zero_shot(t)
    print(f"  '{t}'\n   → {result}\n")

=== ZERO-SHOT RESULTS ===
  'hello there'
   → greeting

  'I have been feeling really anxious lately'
   → asking_mental_health_question

  'thank you so much'
   → gratitude

  'bye goodbye'
   → goodbye

  'what is the best restaurant in Cairo'
   → out_of_scope



# Few shot classifier

In [7]:
def classify_intent_few_shot(user_message: str) -> str:
    
    prompt = f"""You are an intent classification system for a mental health chatbot.

Classify the user message into exactly one of these intents:
greeting, goodbye, gratitude, asking_mental_health_question, out_of_scope

Here are examples for each intent:

greeting:
- "hi"
- "hello how are you"
- "hey good morning"

goodbye:
- "bye"
- "goodbye take care"
- "see you later"

gratitude:
- "thank you so much"
- "thanks, that really helped"
- "I appreciate your support"

asking_mental_health_question:
- "I have been feeling very anxious and can't sleep"
- "how do I deal with depression"
- "I feel hopeless and don't know what to do"
- "my stress levels are very high lately"
- "I had a panic attack today"

out_of_scope:
- "what is the weather today"
- "recommend me a good movie"
- "how do I cook pasta"
- "what is the capital of France"

Rules:
- Reply with ONLY the intent label
- No explanation, no punctuation, just the label

User message: "{user_message}"

Intent:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=20
    )
    
    raw_output = response.choices[0].message.content.strip().lower()
    return raw_output


# Same quick tests
print("=== FEW-SHOT RESULTS ===")
for t in tests:
    result = classify_intent_few_shot(t)
    print(f"  '{t}'\n   → {result}\n")

=== FEW-SHOT RESULTS ===
  'hello there'
   → greeting

  'I have been feeling really anxious lately'
   → asking_mental_health_question

  'thank you so much'
   → gratitude

  'bye goodbye'
   → goodbye

  'what is the best restaurant in Cairo'
   → out_of_scope



# 30 test cases — 6 per intent

In [ ]:
test_data = [
    # greeting
    ("hello", "greeting"),
    ("hi there", "greeting"),
    ("good morning", "greeting"),
    ("hey", "greeting"),
    ("hi how are you", "greeting"),
    ("greetings", "greeting"),

    # goodbye
    ("bye", "goodbye"),
    ("goodbye", "goodbye"),
    ("see you later", "goodbye"),
    ("take care", "goodbye"),
    ("I have to go now", "goodbye"),
    ("farewell", "goodbye"),

    # gratitude
    ("thank you", "gratitude"),
    ("thanks so much", "gratitude"),
    ("I really appreciate it", "gratitude"),
    ("you are very helpful", "gratitude"),
    ("thanks for your support", "gratitude"),
    ("that was really helpful thank you", "gratitude"),

    # asking_mental_health_question
    ("I have been feeling very anxious lately", "asking_mental_health_question"),
    ("how do I deal with depression", "asking_mental_health_question"),
    ("I can't stop crying and I don't know why", "asking_mental_health_question"),
    ("I feel completely hopeless", "asking_mental_health_question"),
    ("my stress is overwhelming me", "asking_mental_health_question"),
    ("I had a panic attack this morning", "asking_mental_health_question"),

    # out_of_scope
    ("what is the weather today", "out_of_scope"),
    ("recommend me a good movie", "out_of_scope"),
    ("how do I cook pasta", "out_of_scope"),
    ("what is 2 plus 2", "out_of_scope"),
    ("who won the world cup", "out_of_scope"),
    ("what is the capital of Egypt", "out_of_scope"),
]

print(f"Total test cases: {len(test_data)}")

Total test cases: 30


In [14]:
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

messages = [item[0] for item in test_data]
true_labels = [item[1] for item in test_data]

print("Running zero-shot predictions...")
zero_shot_preds = [classify_intent_zero_shot(m) for m in messages]

print("Running few-shot predictions...")
few_shot_preds = [classify_intent_few_shot(m) for m in messages]

Running zero-shot predictions...
Running few-shot predictions...


# Prediciton accuracy

In [15]:
zero_acc = accuracy_score(true_labels, zero_shot_preds)
few_acc  = accuracy_score(true_labels, few_shot_preds)

print(f"\n{'='*40}")
print(f"Zero-Shot Accuracy : {zero_acc:.2%}")
print(f"Few-Shot  Accuracy : {few_acc:.2%}")
print(f"{'='*40}")

print("\n--- Zero-Shot Classification Report ---")
print(classification_report(true_labels, zero_shot_preds))

print("\n--- Few-Shot Classification Report ---")
print(classification_report(true_labels, few_shot_preds))


Zero-Shot Accuracy : 100.00%
Few-Shot  Accuracy : 100.00%

--- Zero-Shot Classification Report ---
                               precision    recall  f1-score   support

asking_mental_health_question       1.00      1.00      1.00         6
                      goodbye       1.00      1.00      1.00         6
                    gratitude       1.00      1.00      1.00         6
                     greeting       1.00      1.00      1.00         6
                 out_of_scope       1.00      1.00      1.00         6

                     accuracy                           1.00        30
                    macro avg       1.00      1.00      1.00        30
                 weighted avg       1.00      1.00      1.00        30


--- Few-Shot Classification Report ---
                               precision    recall  f1-score   support

asking_mental_health_question       1.00      1.00      1.00         6
                      goodbye       1.00      1.00      1.00         6
    

# Since both of them did very well on this easy test cases, we will add more complex test cases.

In [27]:
hard_test_data = {
    "Mixed Intent": [
        ("thanks, but I still feel really anxious",            "asking_mental_health_question"),
        ("hello, I have been struggling with depression lately","asking_mental_health_question"),
        ("goodbye, and thank you, I feel much better now",     "goodbye"),
        ("hi, can you help me? I don't know what I'm feeling", "asking_mental_health_question"),
        ("thank you, but honestly I still feel hopeless",      "asking_mental_health_question"),
        ("I appreciate your help but I think I need to go now","goodbye"),
    ],
    "Edge Case": [
        ("help",                          "asking_mental_health_question"),
        ("not great",                     "asking_mental_health_question"),
        ("I don't want to talk about it", "asking_mental_health_question"),
    ],
    "Adversarial": [
        ("my dog died and I feel empty, what food helps with grief",       "asking_mental_health_question"),
        ("what is the weather? I ask because rain makes me depressed",     "asking_mental_health_question"),
        ("I read exercise helps anxiety, what workouts do you recommend",  "asking_mental_health_question"),
        ("my friend is always stressed, what gift should I buy them",      "asking_mental_health_question"),
        ("can you recommend a book about overcoming depression",           "asking_mental_health_question"),
        ("I want to sleep all day, what mattress should I buy",            "out_of_scope"),
        ("how do I help my anxious coworker, we have a meeting tomorrow",  "asking_mental_health_question"),
        ("hello doctor",                                                   "greeting"),
    ],
}

# Flatten while keeping track of category — Python counts, not us
hard_messages, hard_true_labels, category_labels = [], [], []

for category, cases in hard_test_data.items():
    for msg, label in cases:
        hard_messages.append(msg)
        hard_true_labels.append(label)
        category_labels.append(category)

print(f"Total hard cases : {len(hard_messages)}")
for cat, cases in hard_test_data.items():
    print(f"  {cat} : {len(cases)} cases")

Total hard cases : 17
  Mixed Intent : 6 cases
  Edge Case : 3 cases
  Adversarial : 8 cases


# Running zero-shot and few-shot on hard cases

In [28]:
import time

print("Running zero-shot on hard cases...")
hard_zero_preds = []

for i, m in enumerate(hard_messages):
    pred = classify_intent_zero_shot(m)
    hard_zero_preds.append(pred)
    print(f"  [{i+1}/{len(hard_messages)}] '{m[:40]}...' → {pred}")
    time.sleep(0.5)

print("\nDone.")

Running zero-shot on hard cases...
  [1/17] 'thanks, but I still feel really anxious...' → asking_mental_health_question
  [2/17] 'hello, I have been struggling with depre...' → asking_mental_health_question
  [3/17] 'goodbye, and thank you, I feel much bett...' → goodbye
  [4/17] 'hi, can you help me? I don't know what I...' → asking_mental_health_question
  [5/17] 'thank you, but honestly I still feel hop...' → asking_mental_health_question
  [6/17] 'I appreciate your help but I think I nee...' → goodbye
  [7/17] 'help...' → asking_mental_health_question
  [8/17] 'not great...' → asking_mental_health_question
  [9/17] 'I don't want to talk about it...' → asking_mental_health_question
  [10/17] 'my dog died and I feel empty, what food ...' → asking_mental_health_question
  [11/17] 'what is the weather? I ask because rain ...' → asking_mental_health_question
  [12/17] 'I read exercise helps anxiety, what work...' → asking_mental_health_question
  [13/17] 'my friend is always stressed, 

In [30]:
print("Running few-shot on hard cases...")
hard_few_preds = []

for i, m in enumerate(hard_messages):
    pred = classify_intent_few_shot(m)
    hard_few_preds.append(pred)
    print(f"  [{i+1}/{len(hard_messages)}] '{m[:40]}...' → {pred}")
    time.sleep(0.5)

print("\nDone.")

Running few-shot on hard cases...
  [1/17] 'thanks, but I still feel really anxious...' → asking_mental_health_question
  [2/17] 'hello, I have been struggling with depre...' → asking_mental_health_question
  [3/17] 'goodbye, and thank you, I feel much bett...' → gratitude
  [4/17] 'hi, can you help me? I don't know what I...' → asking_mental_health_question
  [5/17] 'thank you, but honestly I still feel hop...' → asking_mental_health_question
  [6/17] 'I appreciate your help but I think I nee...' → goodbye
  [7/17] 'help...' → asking_mental_health_question
  [8/17] 'not great...' → asking_mental_health_question
  [9/17] 'I don't want to talk about it...' → out_of_scope
  [10/17] 'my dog died and I feel empty, what food ...' → asking_mental_health_question
  [11/17] 'what is the weather? I ask because rain ...' → asking_mental_health_question
  [12/17] 'I read exercise helps anxiety, what work...' → asking_mental_health_question
  [13/17] 'my friend is always stressed, what gift ...' →

# Build Results Table

In [31]:
import pandas as pd

hard_results = pd.DataFrame({
    "Category"   : category_labels,
    "Message"    : hard_messages,
    "True Label" : hard_true_labels,
    "Zero-Shot"  : hard_zero_preds,
    "Few-Shot"   : hard_few_preds,
    "Zero ✓"     : [p == t for p, t in zip(hard_zero_preds, hard_true_labels)],
    "Few ✓"      : [p == t for p, t in zip(hard_few_preds,  hard_true_labels)],
})

print(hard_results[["Category","Message","True Label","Zero-Shot","Few-Shot","Zero ✓","Few ✓"]].to_string())

        Category                                                        Message                     True Label                      Zero-Shot                       Few-Shot  Zero ✓  Few ✓
0   Mixed Intent                        thanks, but I still feel really anxious  asking_mental_health_question  asking_mental_health_question  asking_mental_health_question    True   True
1   Mixed Intent           hello, I have been struggling with depression lately  asking_mental_health_question  asking_mental_health_question  asking_mental_health_question    True   True
2   Mixed Intent                 goodbye, and thank you, I feel much better now                        goodbye                        goodbye                      gratitude    True  False
3   Mixed Intent             hi, can you help me? I don't know what I'm feeling  asking_mental_health_question  asking_mental_health_question  asking_mental_health_question    True   True
4   Mixed Intent                  thank you, but honestly I 

# Display Accuracy by Category

In [32]:
from sklearn.metrics import accuracy_score

print("=" * 52)
print(f"{'Category':<20} {'Zero-Shot':>12} {'Few-Shot':>12}")
print("-" * 52)

for cat in ["Mixed Intent", "Edge Case", "Adversarial"]:
    subset = hard_results[hard_results["Category"] == cat]
    z = subset["Zero ✓"].mean()
    f = subset["Few ✓"].mean()
    print(f"{cat:<20} {z:>11.2%} {f:>11.2%}")

print("-" * 52)

z_total = accuracy_score(hard_true_labels, hard_zero_preds)
f_total = accuracy_score(hard_true_labels, hard_few_preds)
print(f"{'TOTAL':<20} {z_total:>11.2%} {f_total:>11.2%}")
print("=" * 52)

Category                Zero-Shot     Few-Shot
----------------------------------------------------
Mixed Intent             100.00%      83.33%
Edge Case                100.00%      66.67%
Adversarial              100.00%      87.50%
----------------------------------------------------
TOTAL                    100.00%      82.35%


# Show the wrong predictions only

In [33]:
print("MISCLASSIFIED BY FEW-SHOT:")
print("=" * 52)

wrong = hard_results[~hard_results["Few ✓"]]

if len(wrong) == 0:
    print("Few-shot got everything correct.")
else:
    for _, row in wrong.iterrows():
        print(f"\n  [{row['Category']}]")
        print(f"  Message    : '{row['Message']}'")
        print(f"  True       : {row['True Label']}")
        print(f"  Few-Shot   : {row['Few-Shot']}  ✗")
        print(f"  Zero-Shot  : {row['Zero-Shot']}  {'✓' if row['Zero ✓'] else '✗'}")

print("\nMISCLASSIFIED BY ZERO-SHOT:")
print("=" * 52)

wrong_z = hard_results[~hard_results["Zero ✓"]]

if len(wrong_z) == 0:
    print("Zero-shot got everything correct.")
else:
    for _, row in wrong_z.iterrows():
        print(f"\n  [{row['Category']}]")
        print(f"  Message    : '{row['Message']}'")
        print(f"  True       : {row['True Label']}")
        print(f"  Zero-Shot  : {row['Zero-Shot']}  ✗")
        print(f"  Few-Shot   : {row['Few-Shot']}  {'✓' if row['Few ✓'] else '✗'}")

MISCLASSIFIED BY FEW-SHOT:

  [Mixed Intent]
  Message    : 'goodbye, and thank you, I feel much better now'
  True       : goodbye
  Few-Shot   : gratitude  ✗
  Zero-Shot  : goodbye  ✓

  [Edge Case]
  Message    : 'I don't want to talk about it'
  True       : asking_mental_health_question
  Few-Shot   : out_of_scope  ✗
  Zero-Shot  : asking_mental_health_question  ✓

  [Adversarial]
  Message    : 'my friend is always stressed, what gift should I buy them'
  True       : asking_mental_health_question
  Few-Shot   : out_of_scope  ✗
  Zero-Shot  : asking_mental_health_question  ✓

MISCLASSIFIED BY ZERO-SHOT:
Zero-shot got everything correct.


# Analysis

### Results Summary
AS Obvious, zero-shot outperformed few-shot on the hard test set.
Zero-shot achieved 100% while few-shot dropped to 82.35%.

### Why Zero-Shot Won
Few-shot examples act as anchors. On simple cases, anchors help.
On complex/mixed cases, they can backfire — the model pattern-matches
to the nearest example rather than reasoning about the full message.
Zero-shot has no anchors, so it reasons purely from the task description,
which turns out to be more flexible for ambiguous inputs.

### Final Decision
**We will use zero-shot as the final classifier.**

as it is:
- 100% on basic test set (30 cases)
- 100% on hard test set including mixed, edge, and adversarial
- More robust on ambiguous inputs
- Simpler prompt = faster API calls = lower latency in deployment

In [36]:
# classify_intent means the zero-shot one, better naming for production use.
def classify_intent(user_message: str) -> str:
    return classify_intent_zero_shot(user_message)
